# Lesson 4.3: Upscaling dataset to 200,000

This notebook is the upscale amount of synthetic customers of the FinFlow Customer Churn Investigation.
The number of customers to upscale to is 200,000


In [1]:
# Imports
from faker import Faker, VERSION
import random
import csv
from datetime import date, timedelta
from pathlib import Path
import pandas as pd
import numpy as np

# Set seeds for reproducibility.
# The number 42 is conventional. Use the same value every time you want
# the same output. Change it if you want a different "random" run.
SEED = 42
Faker.seed(SEED)
random.seed(SEED)

# Pin the dataset to a fixed "as of" date.

# Initialize the Faker instance with US English locale.
# FinFlow's customer base is US-concentrated, so we want US-flavored names,
# addresses, and phone numbers.
fake = Faker('en_US')

# Pin the dataset to a fixed "as of" date.
AS_OF_DATE = date(2026, 1, 1)

# Customers sign up within the 2 years before the as-of date.
SIGNUP_WINDOW_DAYS = 730
SIGNUP_WINDOW_START = AS_OF_DATE - timedelta(days=SIGNUP_WINDOW_DAYS)

print("Faker initialized with seed", SEED, "| Faker version", VERSION)
print("Test draw:", fake.name())

Faker initialized with seed 42 | Faker version 40.38.0
Test draw: Allison Hill


## Adding correlated fields

`monthly_fee` is derived from `plan_tier` (deterministic). `active` is a
weighted boolean (~20% churn rate). `churn_date` is populated only when
`active == False`. Engagement metrics (sessions, duration, support tickets)
correlate with `active` state.


In [2]:
# A lookup from plan_tier to monthly_fee.
# These are the locked FinFlow pricing tiers from the kickoff call.
PLAN_FEES = {
    'Basic': 9.99,
    'Pro': 29.99,
    'Enterprise': 199.00
}

def build_one_customer_v2(index):
    """Build a single FinFlow customer record, version 2 with correlations.

    Args:
        index: integer counter used to generate sequential customer_id.

    Returns:
        Dictionary with all FinFlow customer fields populated.
    """
    first = fake.first_name()
    last = fake.last_name()
    signup_date = SIGNUP_WINDOW_START + timedelta(
        days=random.randint(0, SIGNUP_WINDOW_DAYS)
    )

    # plan_tier with realistic FinFlow distribution
    plan_tier = random.choices(
        ['Basic', 'Pro', 'Enterprise'],
        weights=[60, 30, 10],
        k=1
    )[0]
    monthly_fee = PLAN_FEES[plan_tier]

    # active flag: ~20% of customers are churned
    is_active = random.choices([True, False], weights=[80, 20], k=1)[0]

    # last_login_date and churn_date depend on active state,
    # measured from the pinned as-of date rather than the clock
    today = AS_OF_DATE

    if is_active:
        # Active customers logged in within the last 30 days
        days_since_login = random.randint(0, 30)
        last_login = today - timedelta(days=days_since_login)
        churn_date = None
    else:
        # Churned customers haven't logged in for 30 to 180 days
        days_since_login = random.randint(30, 180)
        last_login = today - timedelta(days=days_since_login)
        # Churn date is somewhere between last_login and today
        days_since_churn = random.randint(0, days_since_login)
        churn_date = today - timedelta(days=days_since_churn)

    # Engagement metrics: correlate with active state
    if is_active:
        total_sessions = random.randint(20, 400)
        avg_session_duration = round(random.uniform(3.0, 15.0), 1)
        support_tickets = random.randint(0, 4)
    else:
        total_sessions = random.randint(1, 50)
        avg_session_duration = round(random.uniform(1.0, 6.0), 1)
        support_tickets = random.randint(0, 8)

    return {
        "customer_id": f"FF-{index:05d}",
        "first_name": first,
        "last_name": last,
        "email": f"{first.lower()}.{last.lower()}{index}@example.com",
        "city": fake.city(),
        "state": fake.state_abbr(),
        "country": "US",
        "signup_date": signup_date.isoformat(),
        "plan_tier": plan_tier,
        "monthly_fee": monthly_fee,
        "last_login_date": last_login.isoformat(),
        "total_sessions": total_sessions,
        "avg_session_duration_minutes": avg_session_duration,
        "support_tickets_opened": support_tickets,
        "active": is_active,
        "churn_date": churn_date.isoformat() if churn_date else None,
    }


# Build and display a single record
sample = build_one_customer_v2(1)
for field, value in sample.items():
    print(f"  {field}: {value}")

  customer_id: FF-00001
  first_name: Jeffrey
  last_name: Doyle
  email: jeffrey.doyle1@example.com
  city: New Roberttown
  state: CO
  country: US
  signup_date: 2025-10-17
  plan_tier: Basic
  monthly_fee: 9.99
  last_login_date: 2025-12-25
  total_sessions: 134
  avg_session_duration_minutes: 4.7
  support_tickets_opened: 0
  active: True
  churn_date: None


## Generating 200,000 customers

Loop the build function. Time it. Inspect the first and last few records.

In [3]:
import time

# Re-seed so the run is reproducible from this point
Faker.seed(SEED)
random.seed(SEED)

# Generate 200,000 customers
start = time.time()
customers = [build_one_customer_v2(i) for i in range(1, 200_001)]
elapsed = time.time() - start

print(f"Built {len(customers)} customers in {elapsed:.2f} seconds")
print()
print("First customer:")
for field, value in customers[0].items():
    print(f"  {field}: {value}")
print()
print("Last customer:")
for field, value in customers[-1].items():
    print(f"  {field}: {value}")

Built 200000 customers in 12.69 seconds

First customer:
  customer_id: FF-00001
  first_name: Danielle
  last_name: Johnson
  email: danielle.johnson1@example.com
  city: Johnberg
  state: WA
  country: US
  signup_date: 2025-10-17
  plan_tier: Basic
  monthly_fee: 9.99
  last_login_date: 2025-12-25
  total_sessions: 134
  avg_session_duration_minutes: 4.7
  support_tickets_opened: 0
  active: True
  churn_date: None

Last customer:
  customer_id: FF-200000
  first_name: Jeffrey
  last_name: Johnson
  email: jeffrey.johnson200000@example.com
  city: Brownview
  state: WY
  country: US
  signup_date: 2024-01-20
  plan_tier: Pro
  monthly_fee: 29.99
  last_login_date: 2025-12-25
  total_sessions: 263
  avg_session_duration_minutes: 6.9
  support_tickets_opened: 1
  active: True
  churn_date: None


## Saving to CSV

Save the generated customers as a CSV file in the lesson folder.

In [4]:
# Define the output path. Path() handles Mac/Windows path differences cleanly.
output_path = Path("finflow-customers-200k.csv")

# Get the field names from the first record (Python 3.7+ dicts preserve order)
field_names = list(customers[0].keys())

# Write the CSV
with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=field_names)
    writer.writeheader()
    writer.writerows(customers)

# Verify
print(f"Wrote {len(customers)} customers to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Wrote 200000 customers to /Users/johnko/northstar-coursework/module-04-python-for-data-analytics/lesson-4-3-pandas-fundamentals/finflow-customers-200k.csv
File size: 25,327,393 bytes


## Set options and check versions for Pandas

Set options in pandas and check Pandas and NumPy version

In [5]:
# Display all DataFrame columns when printing (Pandas truncates by default)
pd.set_option('display.max_columns'loaded, None)

# Show numbers with two decimal places where appropriate
pd.set_option('display.float_format', '{:.2f}'.format)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.0.6
NumPy version: 2.5.3


## Load and Inspect
Load and inspect finflow-customers-200k.csv file into a DataFrame and run a standard inspection routine on the loaded DataFrame

In [6]:
# Load file 
df = pd.read_csv('finflow-customers-200k.csv',
    # Make sure dates are read as dates rather than strings
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

#Check if the number of rows in file to confirm if loaded properly
print(f"Loaded {len(df)} rows from finflow-customers-200k.csv")

Loaded 200000 rows from finflow-customers-k00k.csv


## Inspection routine

The first 40 seconds with any DataFrame. Confirm shape, types, value
ranges, and missing-data profile before asking any analytical question.

In [7]:
# Shape: how many rows and columns?
print("Shape:", df.shape)

Shape: (200000, 16)


In [8]:
# Data types per column
print(df.dtypes)

customer_id                                str
first_name                                 str
last_name                                  str
email                                      str
city                                       str
state                                      str
country                                    str
signup_date                     datetime64[us]
plan_tier                                  str
monthly_fee                            float64
last_login_date                 datetime64[us]
total_sessions                           int64
avg_session_duration_minutes           float64
support_tickets_opened                   int64
active                                    bool
churn_date                      datetime64[us]
dtype: object


In [9]:
# First five rows
df.head()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2025-10-17,Basic,9.99,2025-12-25,134,4.70,0,True,NaT
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-11-24,Pro,29.99,2025-12-14,236,3.40,0,True,NaT
2,FF-00003,Jeffery,Wagner,jeffery.wagner3@example.com,New Jamesside,MT,US,2024-08-12,Basic,9.99,2025-12-15,121,11.60,4,True,NaT
3,FF-00004,Jeffrey,Lawrence,jeffrey.lawrence4@example.com,Port Lindachester,MA,US,2025-03-06,Basic,9.99,2025-12-07,23,12.10,1,True,NaT
4,FF-00005,Joshua,Lewis,joshua.lewis5@example.com,Curtisfurt,CT,US,2025-12-16,Basic,9.99,2025-12-26,192,4.20,3,True,NaT


In [10]:
# Last five rows
df.tail()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date
199995,FF-199996,Mitchell,Johnson,mitchell.johnson199996@example.com,North Ronald,KY,US,2025-06-01,Basic,9.99,2025-12-03,204,7.00,3,True,NaT
199996,FF-199997,Crystal,Thompson,crystal.thompson199997@example.com,Carolynborough,MO,US,2024-10-08,Basic,9.99,2025-12-28,371,10.10,2,True,NaT
199997,FF-199998,Erik,Wiley,erik.wiley199998@example.com,Jeffreyhaven,KS,US,2024-06-05,Basic,9.99,2025-09-10,7,4.90,1,False,2025-10-29
199998,FF-199999,Jonathan,Arias,jonathan.arias199999@example.com,Kellyfurt,NM,US,2024-02-01,Pro,29.99,2025-12-21,346,8.30,4,True,NaT
199999,FF-200000,Jeffrey,Johnson,jeffrey.johnson200000@example.com,Brownview,WY,US,2024-01-20,Pro,29.99,2025-12-25,263,6.90,1,True,NaT


In [11]:
# Full info dump
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   customer_id                   200000 non-null  str           
 1   first_name                    200000 non-null  str           
 2   last_name                     200000 non-null  str           
 3   email                         200000 non-null  str           
 4   city                          200000 non-null  str           
 5   state                         200000 non-null  str           
 6   country                       200000 non-null  str           
 7   signup_date                   200000 non-null  datetime64[us]
 8   plan_tier                     200000 non-null  str           
 9   monthly_fee                   200000 non-null  float64       
 10  last_login_date               200000 non-null  datetime64[us]
 11  total_sessions          

The dataset contains 200,000 rows and 16 columns. All three date columns wer.e converted to datetime values. The missing values in churn_date are expected because active customers do not have a churn date.

In [12]:
# Numeric summary statistics
df.describe()

,signup_date,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,churn_date
count,200000,200000.00,200000,200000.00,200000.00,200000.00,39951
mean,2024-12-31 21:38:44.592000,34.81,2025-11-29 01:01:06.816000,173.01,7.90,2.40,2025-11-09 11:25:24.938049
min,2024-01-02 00:00:00,9.99,2025-07-05 00:00:00,1.00,1.00,0.00,2025-07-05 00:00:00
25%,2024-07-01 00:00:00,9.99,2025-12-03 00:00:00,48.00,4.60,1.00,2025-10-16 00:00:00
50%,2025-01-01 00:00:00,9.99,2025-12-13 00:00:00,162.00,7.50,2.00,2025-11-19 00:00:00
75%,2025-07-03 00:00:00,29.99,2025-12-23 00:00:00,281.00,11.20,4.00,2025-12-11 00:00:00
max,2026-01-01 00:00:00,199.00,2026-01-01 00:00:00,400.00,15.00,8.00,2026-01-01 00:00:00
std,NaN,55.29,NaN,123.05,3.86,1.89,NaN


## Sanity checks

Before saving, verify that the generated data actually matches the
distributions we asked for. Plan tier weights, churn rate, value ranges.

In [13]:
# Check if plan tier distribution is ~ 60%/30%/10%
tier_counts = (df['plan_tier'].value_counts() / len(df)) * 100
print("Expected target: ~60/30/10 \n")
tier_counts

Expected target: ~60/30/10 



plan_tier
Basic        59.94
Pro          30.11
Enterprise    9.95
Name: count, dtype: float64

The data nearly matches all target in 0.11 percentage point or less.

In [14]:
# Check if churn rate is ~20%
churn_counts = df[df['active'] == False]
churn_rate = (len(churn_counts)/len(df)) * 100
print(f"{churn_rate:.2f}%")

19.98%


**Observation:**
Churn rate is
19.98%, within 0.02 point of the 20% target

In [15]:
# Check monthly recurring revenue from active customers
active_df = df[df["active"] == True]
total_mmr = active_df["monthly_fee"].sum()
print(f"Total MRR (only active customers): ${total_mmr:,.2f}")

Total monthly revenue (only active customers): $5,573,532.87


In [16]:
# Check signup date range
signup_df = df["signup_date"]
print(f"\nSignup date range: {signup_df.min().date()} to {signup_df.max().date()}")


Signup date range: 2024-01-02 to 2026-01-01


Signup dates span the
intended 2-year window. Ready to save and continue.

**Observation** Generating 200,000 records took approximately 12.69 seconds, compared with 0.07 seconds for the 1,000-row dataset.
With 200,000 records, the random variation has less influence on the overall percentages. Large sample size generally produces proportions closer to the intended distribution. 
The load time for the larger CSV was not noticeable different to the load time to 1,000-record CSV.